# 01 · KARD dataset exploration

All numbers below are computed from the extracted files (`data/interim/kard_realworld.npz`, built by `make prepare-data`).

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from skeleton_action.constants import CLASS_NAMES, JOINT_NAMES, KARD_EDGES, J
from skeleton_action.data.kard import KardCollection
INTERIM = ROOT / "data/interim/kard_realworld.npz"
assert INTERIM.exists(), "run `make data` first"
col = KardCollection.load(INTERIM)
print(len(col), "sequences")

## Inspection report (generated by `make inspect-data`)

In [ ]:
rep_path = ROOT / "outputs/reports/kard_inspection_realworld.json"
if rep_path.exists():
    rep = json.loads(rep_path.read_text())
    print({k: rep[k] for k in ("files_found", "valid_sequences", "expected_sequences")})
    print("malformed:", len(rep["malformed_sequences"]), "missing:", len(rep["missing_sequences"]),
          "duplicates:", len(rep["duplicates"]))
else:
    print("no report yet: run `make inspect-data`")

## Class and subject distribution

In [ ]:
meta = pd.DataFrame({"seq_id": col.seq_ids, "action": col.actions, "subject": col.subjects,
                     "rep": col.repetitions, "frames": col.lengths})
meta["label"] = [CLASS_NAMES[a - 1] for a in meta.action]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
meta.label.value_counts().reindex(CLASS_NAMES).plot.bar(ax=ax[0], title="sequences per action")
meta.subject.value_counts().sort_index().plot.bar(ax=ax[1], title="sequences per subject")
plt.tight_layout(); plt.show()
pd.crosstab(meta.label, meta.subject).loc[CLASS_NAMES]

## Sequence length per action

Lengths vary by a factor ~2.5 within and across classes (Walk and Phone call are the longest). The models see every sequence resampled to a fixed T, so absolute duration is not an input feature.

In [ ]:
order = meta.groupby("label").frames.median().sort_values().index
fig, ax = plt.subplots(figsize=(12, 4))
ax.boxplot([meta[meta.label == c].frames for c in order], tick_labels=order)
ax.set_ylabel("frames"); plt.xticks(rotation=60, ha="right"); plt.tight_layout(); plt.show()
meta.frames.describe()

## Coordinate ranges (real-world, mm)

In [ ]:
c = col.coords
pd.DataFrame({ax: pd.Series(c[..., i].ravel()).describe() for i, ax in enumerate("xyz")}).T

In [ ]:
per_joint = pd.DataFrame({ax: [c[:, j, i].mean() for j in range(15)] for i, ax in enumerate("xyz")},
                         index=JOINT_NAMES)
per_joint.round(0)

## Which limb moves? (convention check used for the Kinect mapping)

In [ ]:
rows = []
for a in range(1, 19):
    idx = np.flatnonzero(col.actions == a)
    def path(j):
        return np.median([np.linalg.norm(np.diff(col.sequence(i)[:, j] - col.sequence(i)[:, J.TORSO], axis=0), axis=1).sum() for i in idx])
    rows.append({"action": CLASS_NAMES[a - 1], "R hand": path(J.R_HAND), "L hand": path(J.L_HAND),
                 "R foot": path(J.R_FOOT), "L foot": path(J.L_FOOT)})
pd.DataFrame(rows).set_index("action").round(0)